<table style="border:0; background:none; width:100%">
<tr>
<td style="vertical-align: middle; width: 60%;">
<b>Pontificia Universidad Javeriana</b><br/>
Facultad de Ingeniería · Departamento de Ingeniería de Sistemas<br/>
<b>Procesamiento de Datos a Gran Escala</b>
</td>
<td style="vertical-align: middle; text-align: right; width: 40%;">
<b>Proyecto de Investigación — Entrega 2</b><br/>
Brecha digital y resultados Saber 11 en Colombia<br/>
<b>Grupo: REST pAPIs</b>
</td>
</tr>
</table>

---

# 03 — Silver: Internet Fijo (MinTIC)

Accesos a internet fijo por municipio, año, trimestre, proveedor y tecnología.

## FILTROS APLICADOS

| # | Filtro | Justificación |
|---|---|---|
| F1 | `NUM_ACCESOS IS NOT NULL AND NUM_ACCESOS >= 0` | Registros sin accesos o con valores negativos son errores del reporte trimestral MinTIC; no aportan información medible. |
| F2 | `VELOCIDAD_BAJADA > 0` | Velocidad de bajada 0 Mbps es físicamente imposible para un servicio activo; identifica registros malformados o de servicios suspendidos no marcados como tal. |

## TRANSFORMACIONES APLICADAS

| # | Transformación | Justificación |
|---|---|---|
| T1 | Renombrar `AÑO` → `ANO`, `No DE ACCESOS` → `NUM_ACCESOS` | El carácter `Ñ` y los espacios en nombres de columna rompen selección por nombre en Spark/HDFS. |
| T2 | Cast `ANO`, `TRIMESTRE` → IntegerType ; `NUM_ACCESOS` → LongType | Bronze los preserva como string; sin tipado numérico no se pueden agregar (sum, avg). |
| T3 | Parsear `VELOCIDAD_BAJADA`/`VELOCIDAD_SUBIDA`: `regexp_replace(',', '.')` → DoubleType | Los valores vienen con coma decimal (locale es-CO, ej. `"8,00"`); cast directo a double da null. |
| T4 | `lpad(COD_DEPARTAMENTO, 2, '0')` → `COD_DEPTO`; `lpad(COD_MUNICIPIO, 5, '0')` → `COD_MPIO` | Códigos DANE son claves estándar de 2 y 5 dígitos. Sin zero-padding el join contra ICFES/SISBEN/MEN falla (códigos como `5` vs `05`). |
| T5 | Normalizar `DEPARTAMENTO`, `MUNICIPIO`, `PROVEEDOR`, `TECNOLOGIA`: `UPPER` + `translate` de acentos | Evita duplicación lógica de categorías por diferencias de mayúscula/minúscula o acentos (ej. `Antioquia` vs `ANTIOQUIA` vs `ANTIOQUÍA`). |
| T6 | Selección de columnas finales y `partitionBy(ANO)` al escribir | Reduce I/O en consultas filtradas por año (3-5× más rápido). |

## 1. Setup y carga

In [1]:
import sys
sys.path.insert(0, "/home/estudiante/proyecto_datos/scripts")
from common_spark import get_spark, P
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, DoubleType

spark = get_spark("Entrega2-Silver-Internet", executor_memory="3g", driver_memory="2g", cores=2)
df = spark.read.parquet(P.BRONZE_PQ_INTERNET)
print(f"Filas: {df.count():,}   Cols: {len(df.columns)}")
df.printSchema()
df.show(3, truncate=False)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/22 08:13:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Filas: 2,795,052   Cols: 12
root
 |-- AÑO: string (nullable = true)
 |-- TRIMESTRE: string (nullable = true)
 |-- PROVEEDOR: string (nullable = true)
 |-- COD_DEPARTAMENTO: string (nullable = true)
 |-- DEPARTAMENTO: string (nullable = true)
 |-- COD_MUNICIPIO: string (nullable = true)
 |-- MUNICIPIO: string (nullable = true)
 |-- SEGMENTO: string (nullable = true)
 |-- TECNOLOGIA: string (nullable = true)
 |-- VELOCIDAD_BAJADA: string (nullable = true)
 |-- VELOCIDAD_SUBIDA: string (nullable = true)
 |-- No DE ACCESOS: string (nullable = true)



+----+---------+-------------------------------------+----------------+------------+-------------+--------------------+-----------+----------+----------------+----------------+-------------+
|AÑO |TRIMESTRE|PROVEEDOR                            |COD_DEPARTAMENTO|DEPARTAMENTO|COD_MUNICIPIO|MUNICIPIO           |SEGMENTO   |TECNOLOGIA|VELOCIDAD_BAJADA|VELOCIDAD_SUBIDA|No DE ACCESOS|
+----+---------+-------------------------------------+----------------+------------+-------------+--------------------+-----------+----------+----------------+----------------+-------------+
|2018|1        |EDATEL S.A.                          |05              |ANTIOQUIA   |05042        |SANTAFÉ DE ANTIOQUIA|CORPORATIVO|XDSL      |8,00            |1,00            |25           |
|2019|1        |AXESS NETWORKS SOLUTIONS COLOMBIA SAS|52              |NARIÑO      |52473        |MOSQUERA            |CORPORATIVO|SATELITAL |0,06            |0,06            |2            |
|2019|1        |TELMEX COLOMBIA S.A.         

## 2. Renombres y tipos

In [2]:
ACCENTS_FROM = "ÁÉÍÓÚÜÑáéíóúüñ"
ACCENTS_TO   = "AEIOUUNaeiouun"

df_s = (
    df
    # Renombrar columna molesta
    .withColumnRenamed("AÑO", "ANO")
    .withColumnRenamed("No DE ACCESOS", "NUM_ACCESOS")
    # Tipos
    .withColumn("ANO", F.col("ANO").cast(IntegerType()))
    .withColumn("TRIMESTRE", F.col("TRIMESTRE").cast(IntegerType()))
    .withColumn("NUM_ACCESOS", F.col("NUM_ACCESOS").cast(LongType()))
    # Velocidades: coma decimal → punto
    .withColumn("VELOCIDAD_BAJADA", F.regexp_replace(F.col("VELOCIDAD_BAJADA"), ",", ".").cast(DoubleType()))
    .withColumn("VELOCIDAD_SUBIDA", F.regexp_replace(F.col("VELOCIDAD_SUBIDA"), ",", ".").cast(DoubleType()))
    # Códigos geográficos
    .withColumn("COD_DEPTO", F.lpad(F.col("COD_DEPARTAMENTO"), 2, "0"))
    .withColumn("COD_MPIO",  F.lpad(F.col("COD_MUNICIPIO"), 5, "0"))
    # Normalización de nombres
    .withColumn("DEPARTAMENTO", F.upper(F.translate(F.col("DEPARTAMENTO"), ACCENTS_FROM, ACCENTS_TO)))
    .withColumn("MUNICIPIO",    F.upper(F.translate(F.col("MUNICIPIO"),    ACCENTS_FROM, ACCENTS_TO)))
    .withColumn("PROVEEDOR",    F.upper(F.translate(F.col("PROVEEDOR"),    ACCENTS_FROM, ACCENTS_TO)))
    .withColumn("SEGMENTO",     F.upper(F.col("SEGMENTO")))
    .withColumn("TECNOLOGIA",   F.upper(F.translate(F.col("TECNOLOGIA"),   ACCENTS_FROM, ACCENTS_TO)))
)
df_s.printSchema()
df_s.select("ANO","TRIMESTRE","COD_MPIO","MUNICIPIO","TECNOLOGIA","VELOCIDAD_BAJADA","NUM_ACCESOS").show(5, truncate=False)

root
 |-- ANO: integer (nullable = true)
 |-- TRIMESTRE: integer (nullable = true)
 |-- PROVEEDOR: string (nullable = true)
 |-- COD_DEPARTAMENTO: string (nullable = true)
 |-- DEPARTAMENTO: string (nullable = true)
 |-- COD_MUNICIPIO: string (nullable = true)
 |-- MUNICIPIO: string (nullable = true)
 |-- SEGMENTO: string (nullable = true)
 |-- TECNOLOGIA: string (nullable = true)
 |-- VELOCIDAD_BAJADA: double (nullable = true)
 |-- VELOCIDAD_SUBIDA: double (nullable = true)
 |-- NUM_ACCESOS: long (nullable = true)
 |-- COD_DEPTO: string (nullable = true)
 |-- COD_MPIO: string (nullable = true)



+----+---------+--------+--------------------+----------+----------------+-----------+
|ANO |TRIMESTRE|COD_MPIO|MUNICIPIO           |TECNOLOGIA|VELOCIDAD_BAJADA|NUM_ACCESOS|
+----+---------+--------+--------------------+----------+----------------+-----------+
|2018|1        |05042   |SANTAFE DE ANTIOQUIA|XDSL      |8.0             |25         |
|2019|1        |52473   |MOSQUERA            |SATELITAL |0.06            |2          |
|2019|1        |25269   |FACATATIVA          |CABLE     |40.0            |55         |
|2019|1        |08685   |SANTO TOMAS         |XDSL      |8.0             |1          |
|2020|2        |41551   |PITALITO            |WIFI      |2.5             |37         |
+----+---------+--------+--------------------+----------+----------------+-----------+
only showing top 5 rows



## 3. Calidad: nulos por columna

In [3]:
nulos = df_s.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_s.columns
]).toPandas().T
nulos.columns = ["nulls"]
nulos["pct"] = (nulos["nulls"] / df_s.count() * 100).round(2)
nulos.sort_values("nulls", ascending=False)

,nulls,pct
COD_MUNICIPIO,10,0.0
COD_MPIO,10,0.0
COD_DEPTO,5,0.0
COD_DEPARTAMENTO,5,0.0
TRIMESTRE,0,0.0
ANO,0,0.0
DEPARTAMENTO,0,0.0
PROVEEDOR,0,0.0
MUNICIPIO,0,0.0
SEGMENTO,0,0.0


## 4. Aplicación de filtros (F1 + F2)

In [4]:
n0 = df_s.count()
print(f"Filas antes de filtros: {n0:,}")

# F1 — accesos válidos (no nulos, no negativos)
df_s = df_s.filter(F.col("NUM_ACCESOS").isNotNull() & (F.col("NUM_ACCESOS") >= 0))
n1 = df_s.count()
print(f"  Tras F1 (NUM_ACCESOS válido)              : {n1:>10,}  (eliminó {n0-n1:,})")

# F2 — velocidad de bajada estrictamente positiva
df_s = df_s.filter(F.col("VELOCIDAD_BAJADA").isNotNull() & (F.col("VELOCIDAD_BAJADA") > 0))
n2 = df_s.count()
print(f"  Tras F2 (VELOCIDAD_BAJADA > 0)            : {n2:>10,}  (eliminó {n1-n2:,})")
print(f"Total eliminado por filtros: {n0-n2:,} ({100*(n0-n2)/n0:.2f}%)")

Filas antes de filtros: 2,795,052


  Tras F1 (NUM_ACCESOS válido)              :  2,795,052  (eliminó 0)


  Tras F2 (VELOCIDAD_BAJADA > 0)            :  2,792,934  (eliminó 2,118)
Total eliminado por filtros: 2,118 (0.08%)


## 5. Escribir Silver Parquet (particionado por ANO)

In [5]:
import time
COLS_FINAL = [
    "ANO","TRIMESTRE","PROVEEDOR",
    "COD_DEPTO","DEPARTAMENTO",
    "COD_MPIO","MUNICIPIO",
    "SEGMENTO","TECNOLOGIA",
    "VELOCIDAD_BAJADA","VELOCIDAD_SUBIDA",
    "NUM_ACCESOS",
]
t0 = time.time()
(df_s.select(*COLS_FINAL).write
    .mode("overwrite")
    .partitionBy("ANO")
    .option("compression","snappy")
    .parquet(P.SILVER_INTERNET))
print(f"Escrito en {time.time()-t0:.1f}s")

Escrito en 39.1s


## 6. Verificación

In [6]:
sv = spark.read.parquet(P.SILVER_INTERNET)
print(f"Silver rows: {sv.count():,}")
sv.groupBy("ANO").agg(F.sum("NUM_ACCESOS").alias("total_accesos"),
                     F.countDistinct("COD_MPIO").alias("municipios_distintos")) \
  .orderBy("ANO").show()

Silver rows: 2,792,934


+----+-------------+--------------------+
| ANO|total_accesos|municipios_distintos|
+----+-------------+--------------------+
|2016|          463|                   8|
|2017|     21252904|                1106|
|2018|     26451239|                1111|
|2019|     27712863|                1117|
|2020|     30219937|                1118|
|2021|     33274263|                1118|
|2022|     35172068|                1118|
|2023|     26978816|                1118|
+----+-------------+--------------------+



In [7]:
spark.stop()

26/05/22 08:15:14 WARN NioEventLoop: Selector.select() returned prematurely 512 times in a row; rebuilding Selector io.netty.channel.nio.SelectedSelectionKeySetSelector@40b97cde.
26/05/22 08:15:14 WARN NioEventLoop: Selector.select() returned prematurely 512 times in a row; rebuilding Selector io.netty.channel.nio.SelectedSelectionKeySetSelector@24ed2a30.
